In [ ]:
# importing necessary libraries for MySQL connection and error handling
import mysql.connector
from mysql.connector import Error

import pandas as pd

try:
    # connecting to MySQL server
    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="", 
        allow_local_infile=True
    )
    
    # creating a cursor object to execute SQL queries
    cursor = connection.cursor()

    # if the database already exists, drop it and create a new one (mostly becuase of testing)
    cursor.execute("DROP DATABASE IF EXISTS techreads_db")

    # creates the database and uses it
    cursor.execute("CREATE DATABASE techreads_db")
    cursor.execute("USE techreads_db")

    # creates the listings table with appropriate data types
    cursor.execute("""
        CREATE TABLE listings (
            listing_id BIGINT PRIMARY KEY,
            title VARCHAR(600),
            author VARCHAR(255),
            year YEAR,
            price DECIMAL(10,2),
            currency CHAR(3),
            seller_rating INT
        )
    """)

    # loading the data from the CSV file into the listings table
    # if running from your own, make sure to update the file path to where your CSV file is located
    load_query = """
        LOAD DATA LOCAL INFILE 'C:/Users/bakht/Desktop/DE/Code/Task2_csvToSQL/techreads_books.csv'
        INTO TABLE listings
        FIELDS TERMINATED BY ','
        ENCLOSED BY '"'
        LINES TERMINATED BY '\\n'
        IGNORE 1 ROWS
        (@dummy, listing_id, title, author, year, price, currency, seller_rating)
    """

    # executing the load query and committing the transaction to save changes to the database
    cursor.execute(load_query)
    connection.commit()

    # printing a success message after loading the CSV data into the MySQL database
    print("CSV successfully loaded into MySQL.")

    # query to select the title, price, and seller rating of the listings, ordered by price in descending order
    cursor.execute("""
        SELECT title, price, seller_rating
        FROM listings
        ORDER BY price DESC
    """)

    # fetching all results from the executed query
    results = cursor.fetchall()

    # create dataframe from the results for better visualization and analysis
    df = pd.DataFrame(results, columns=['Title', 'Price', 'Seller Rating'])
    print("\nDataFrame of Results:")
    print(df.head(15))  # printing the top 15 rows of the dataframe


# handling any errors that may occur during the database connection and operations
except Error as e:
    print("Error:", e)

# ensuring that the database connection is closed properly after operations are completed, even if an error occurs
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("\nMySQL connection closed.")

CSV successfully loaded into MySQL.

DataFrame of Results:
                                                Title  Price  Seller Rating
0                   DATA DRIVEN SCIENCE & ENGINEERING  49.45              5
1   Data Engineering with AWS - Second Edition: Ac...  43.29              5
2   Data Engineering Design Patterns : Recipes for...  41.46              5
3   Financial Data Engineering : Design and Build ...  37.61              5
4   Data Engineering with Python: Work with massiv...  27.58              5
5   Ace the Data Engineering Interview: Questions ...  23.63              5
6   Data Engineering with Google Cloud Platform: A...  22.80              5
7         Engineering Tables and Data Classic Reprint  22.58              4
8                            Data-Centric Engineering  22.45              4
9   Local Engineering Data for St Louis Classic Re...  22.36              4
10  Engineering Principles and Practical Data Rela...  22.35              4
11  Regularized System Identi